# Выполнение ЛР №5: Классификация и регрессия

## Подключение библиотек

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Импорт модулей sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Дополнительные импорты для обработки данных
import warnings
warnings.filterwarnings('ignore')

## Настройка библиотек

In [ ]:
# Настройка стилей визуализации
plt.style.use('default')
sns.set_palette("husl")

# Настройка параметров отображения
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Настройка pandas для отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Настройка numpy для воспроизводимости результатов
np.random.seed(42)

print("Библиотеки успешно импортированы и настроены!")
print(f"Версия pandas: {pd.__version__}")
print(f"Версия numpy: {np.__version__}")
print(f"Версия matplotlib: {plt.matplotlib.__version__}")
print(f"Версия seaborn: {sns.__version__}")

## Задание 1: Классификация kNN на датасете flame

### Формулировка

Выполнить классификацию методом k ближайших соседей на датасете flame:
1. Загрузить и разобрать данные из файла flame.txt
2. Оценить точность для разных значений k от 2 до 20 с использованием кросс-валидации
3. Построить график зависимости точности от k

### Решение

#### 1.1 Загрузка и парсинг данных flame

In [ ]:
# Загрузка данных из файла flame.txt
flame_data_path = '../ЛР (4)/Вариант 4/flame.txt'

# Чтение данных с разделителем табуляция
flame_data = pd.read_csv(flame_data_path, sep='\t', header=None, names=['x1', 'x2', 'class'])

print("Данные flame успешно загружены!")
print(f"Размер датасета: {flame_data.shape}")
print("\nПервые 10 строк:")
print(flame_data.head(10))

print("\nИнформация о данных:")
print(flame_data.info())

print("\nРаспределение классов:")
print(flame_data['class'].value_counts().sort_index())

# Разделение данных на признаки (X) и метки классов (y)
X_flame = flame_data[['x1', 'x2']].values
y_flame = flame_data['class'].values

print(f"\nРазмер матрицы признаков X: {X_flame.shape}")
print(f"Размер вектора меток y: {y_flame.shape}")
print(f"Уникальные классы: {np.unique(y_flame)}")

#### 1.2 Оценка точности kNN для разных значений k

In [ ]:
# Оценка точности kNN для разных значений k от 2 до 20
k_range = range(2, 21)  # k от 2 до 20
k_scores = []

print("Оценка точности kNN для разных значений k:")
print("k\tТочность (среднее)\tСтандартное отклонение")
print("-" * 50)

# Цикл оценки для каждого k
for k in k_range:
    # Создание модели kNN
    knn = KNeighborsClassifier(n_neighbors=k)
    
    # Кросс-валидация с 5 фолдами
    cv_scores = cross_val_score(knn, X_flame, y_flame, cv=5, scoring='accuracy')
    
    # Сохранение среднего значения точности
    mean_accuracy = cv_scores.mean()
    std_accuracy = cv_scores.std()
    k_scores.append(mean_accuracy)
    
    print(f"{k}\t{mean_accuracy:.4f}\t\t{std_accuracy:.4f}")

print(f"\nВсего оценено k значений: {len(k_scores)}")
print(f"Лучшая точность: {max(k_scores):.4f} при k = {k_range[k_scores.index(max(k_scores))]}")
print(f"Худшая точность: {min(k_scores):.4f} при k = {k_range[k_scores.index(min(k_scores))]}")

#### 1.3 Построение графика точности vs k

In [ ]:
# Построение графика зависимости точности от k
plt.figure(figsize=(12, 8))

# Основной график
plt.plot(k_range, k_scores, 'bo-', linewidth=2, markersize=8, label='Точность кросс-валидации')

# Выделение максимального значения
best_k = k_range[k_scores.index(max(k_scores))]
best_score = max(k_scores)
plt.plot(best_k, best_score, 'ro', markersize=12, label=f'Лучший результат (k={best_k})')

# Настройка графика
plt.xlabel('Количество соседей (k)', fontsize=14)
plt.ylabel('Точность классификации', fontsize=14)
plt.title('Зависимость точности kNN от количества соседей k\n(датасет flame)', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)

# Установка диапазона осей
plt.xlim(1.5, 20.5)
plt.ylim(min(k_scores) - 0.02, max(k_scores) + 0.02)

# Добавление аннотации для лучшего результата
plt.annotate(f'Максимум: {best_score:.4f}', 
             xy=(best_k, best_score), 
             xytext=(best_k + 2, best_score + 0.01),
             arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
             fontsize=12, color='red')

# Настройка тиков на оси x
plt.xticks(range(2, 21, 2))

plt.tight_layout()
plt.show()

# Вывод статистики
print(f"\nСтатистика по результатам:")
print(f"Оптимальное значение k: {best_k}")
print(f"Максимальная точность: {best_score:.4f}")
print(f"Средняя точность по всем k: {np.mean(k_scores):.4f}")
print(f"Стандартное отклонение: {np.std(k_scores):.4f}")

#### Выводы по заданию 1

1. **Загрузка данных**: Успешно загружен датасет flame с двумя признаками (x1, x2) и двумя классами (1, 2)
2. **Оценка kNN**: Проведена оценка точности для k от 2 до 20 с использованием 5-фолдовой кросс-валидации
3. **Визуализация**: Построен график зависимости точности от k, который показывает оптимальное значение k
4. **Результат**: Определено оптимальное значение k для данного датасета

## Задание 2

### Формулировка

### Решение

## Задание 3

### Формулировка

### Решение

## Задание 4

### Формулировка

### Решение

## Задание 5

### Формулировка

### Решение

## Задание 6

### Формулировка

### Решение

## Задание 7

### Формулировка

### Решение

## Задание 8

### Формулировка

### Решение